## Cell 1: Install Dependencies

In [ ]:
!pip install supar openai sentencepiece wandb nltk spacy huggingface_hub rouge-score bert-score numpy language_tool_python==2.9.2 scikit-learn scipy tqdm bitsandbytes transformers Pillow ftfy regex flake8 yapf isort yacs gdown tb-nightly future wilds tabulate torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --extra-index-url https://download.pytorch.org/whl/cu124


## Cell 2: Create Logs Directory

In [ ]:
import os

path = "/kaggle/working/logs"
os.makedirs(path, exist_ok=True)
print(f"Directory created at: {path}")


## Cell 3: Imports, Args, HF Login, Model Setup, WandB

In [ ]:
#!/usr/bin/env python

import time, gc, os, re, json, random, string, heapq, logging, argparse
import numpy as np
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.tokenize.treebank import TreebankWordDetokenizer
from scipy.stats import entropy
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import difflib
from nltk.metrics.distance import edit_distance
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

from tqdm import tqdm
import spacy
import language_tool_python

# ── Argument Parsing ──────────────────────────────────────────────────────────
parser = argparse.ArgumentParser(description='Multi-objective Genetic Search for Prompt Optimization')
parser.add_argument('--mode', default="Instruction Only", help='Mode of instructions/prompts')
parser.add_argument('--num-shots', default=2, type=int)
parser.add_argument('--batch-size', default=1, type=int)
parser.add_argument('--task-idx', default=0, type=int)
parser.add_argument('--seed', default=3, type=int)
parser.add_argument('--train-seed', default=42, type=int)
parser.add_argument('--num-compose', default=1, type=int)
parser.add_argument('--num-train', default=50, type=int, help='Reduced for T4 budget')
parser.add_argument('--level', default="phrase")
parser.add_argument('--simulated-anneal', action='store_true', default=False)
parser.add_argument('--agnostic', action='store_true', default=False)
parser.add_argument('--print-orig', action='store_true', default=True)
parser.add_argument('--write-preds', action='store_true', default=True)
parser.add_argument('--meta-dir', default='/kaggle/working/')
parser.add_argument('--meta-name', default='meta_output.txt')
parser.add_argument('--patience', default=3, type=int)
parser.add_argument('--num-candidates', default=5, type=int)
parser.add_argument('--num-iter', default=5, type=int)
parser.add_argument('--key-id', default=None, type=int)
parser.add_argument('--edits', nargs="+", default=['del', 'swap', 'sub', 'add'])
parser.add_argument('--tournament-selection', default=2, type=int)
parser.add_argument('--population-size', default=10, type=int)
parser.add_argument('--num-offspring', default=2, type=int)
parser.add_argument('--mutation-prob', default=0.5, type=float)
parser.add_argument('--data-dir', default='/kaggle/input/datasets/anshumanmoharana07/biolaysumm/Data/Short')
parser.add_argument('--project-name', default='biolay-moga-ministral')
parser.add_argument('--budget', default=8000, type=int)
args, unknown = parser.parse_known_args()

# Clear score files
for fname in ['rouge_scores.txt', 'bert_scores.txt', 'bleu_scores.txt']:
    open(os.path.join(args.meta_dir, fname), 'w').close()

tool = language_tool_python.LanguageTool('en-US')

global_progress_bar = tqdm(total=args.budget, desc="API Calls", leave=False, position=0, dynamic_ncols=True)

# ── WandB ─────────────────────────────────────────────────────────────────────
try:
    import wandb
    wandb.login(key='')  # add your wandb key here
    wandb.init(project=args.project_name)
    wandb.config.update(args)
    tqdm.write("Wandb is setup\n")
except Exception as e:
    tqdm.write(f"Error while init wandb: {e}\n")

# ── HuggingFace Login (required for Ministral — gated model) ──────────────────
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

# ── 4-bit Quantized Ministral-8B ──────────────────────────────────────────────
torch.random.manual_seed(0)

model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

try:
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True
    )
except RuntimeError as e:
    if "CUDA out of memory" in str(e):
        print("CUDA OOM, clearing cache and retrying...")
        torch.cuda.empty_cache()
        gc.collect()
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
            low_cpu_mem_usage=True
        )

tokenizer = AutoTokenizer.from_pretrained(model_name)

generation_args = {
    "max_new_tokens": 150,
    "temperature": 0.1,
    "do_sample": False
}

print("GPU Utilization:")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.memory_allocated(i)/1024**3:.2f} GB allocated, "
          f"{torch.cuda.memory_reserved(i)/1024**3:.2f} GB reserved")

evaluation_cache = {}


## Cell 4: Inference Function (complete_phi4)

In [ ]:
def counter(func):
    def wrapper(*args, **kwargs):
        wrapper.count += 1
        global_progress_bar.update(1)
        return func(*args, **kwargs)
    wrapper.count = 0
    return wrapper


@counter
def complete_phi4(prompt, max_tokens=None):
    """
    Inference function for Ministral-8B.
    NOTE: enable_thinking=False removed (Qwen-specific argument not supported by Ministral).
    """
    messages = prompt
    args_local = generation_args.copy()
    if max_tokens:
        args_local["max_new_tokens"] = max_tokens

    # Apply chat template — no enable_thinking for Ministral
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=args_local["max_new_tokens"],
            temperature=args_local["temperature"],
            do_sample=args_local["do_sample"]
        )

    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
    content = tokenizer.decode(output_ids, skip_special_tokens=True).strip("\n")
    return content


## Cell 5: TXT Data Loader for BioLaySumm Short Dataset

In [ ]:
import glob

def load_biolay_txt_dataset(data_dir):
    """
    Loads BioLaySumm Short dataset from .txt files.
    Pairs fulltext and summaries by numeric ID extracted from filename.
    Fulltext: multiclinsum_gs_en_<ID>.txt
    Summary:  multiclinsum_gs_en_<ID>_sum.txt
    """
    fulltext_dir = os.path.join(data_dir, 'fulltext')
    summary_dir  = os.path.join(data_dir, 'summaries')

    fulltext_map = {}
    for fpath in glob.glob(os.path.join(fulltext_dir, '*.txt')):
        fname = os.path.basename(fpath)
        match = re.match(r'.*_(\d+)\.txt$', fname)
        if match and '_sum' not in fname:
            fulltext_map[match.group(1)] = fpath

    summary_map = {}
    for fpath in glob.glob(os.path.join(summary_dir, '*.txt')):
        fname = os.path.basename(fpath)
        match = re.match(r'.*_(\d+)_sum\.txt$', fname)
        if match:
            summary_map[match.group(1)] = fpath

    common_ids = sorted(set(fulltext_map.keys()) & set(summary_map.keys()), key=lambda x: int(x))
    print(f"Found {len(common_ids)} paired instances (fulltext + summary)")

    instances = []
    for id_ in common_ids:
        with open(fulltext_map[id_], 'r', encoding='utf-8') as f:
            article = f.read().strip()
        with open(summary_map[id_], 'r', encoding='utf-8') as f:
            summary = f.read().strip()
        instances.append({'input': article, 'output': summary})

    positive_examples = [
        {'input': inst['input'], 'output': inst['output']}
        for inst in instances[:2]
    ]
    data = {
        'Definition': 'Write a concise clinical summary of the following case report.',
        'Positive Examples': positive_examples,
        'Instances': instances[2:]
    }
    return data


nltk.download('punkt')
nltk.download('punkt_tab')

DATASET = load_biolay_txt_dataset(args.data_dir)
print(f"Total instances available for train/test: {len(DATASET['Instances'])}")


## Cell 6: Setup Variables + Encode / Prompt Construction Functions

In [ ]:
meta_path = os.path.join(args.meta_dir, args.meta_name)
meta_file = open(meta_path, 'w+')
batch_size      = args.batch_size
num_shots       = args.num_shots
mode            = args.mode
seed            = args.seed
train_seed      = args.train_seed
num_samples     = 100
num_train_samples = args.num_train

np.random.seed(args.train_seed)
torch.manual_seed(args.train_seed)
population_size  = args.population_size
num_compose      = args.num_compose
num_candidates   = args.num_candidates
num_steps        = args.num_iter
num_tournaments  = args.tournament_selection
T_max            = 10
edit_operations  = args.edits
use_add          = 'add' in args.edits
num_offspring    = args.num_offspring
mutation_prob    = args.mutation_prob

if 'wandb' in globals():
    wandb.log({"Num Compose": num_compose, "Num Candidates": num_candidates,
               "Max Iterations": num_steps, "Tournament Selections": num_tournaments,
               "Edit Operations": edit_operations, "Population Size": population_size,
               "Num Offspring": num_offspring, "Patience": args.patience,
               "Mutation Probability": mutation_prob})


def encode_instruction(instruction_structure=['Definition'], number_of_examples=0,
                       number_of_instances=100, null_word=None, data_seed=0,
                       modified={}, args=args):
    random.seed(0)
    instances = DATASET['Instances']
    instance_indices = list(range(len(instances)))
    test_indices = random.sample(instance_indices, min(number_of_instances, len(instances)))

    random.seed(data_seed)
    pos_examples = DATASET['Positive Examples']
    chosen_examples = random.sample(pos_examples, min(number_of_examples, len(pos_examples))) \
                      if number_of_examples > 0 else []

    definition = modified.get('Definition', DATASET['Definition'])
    generic_instruction = 'Definition: ' + definition.strip()
    if 'Positive Examples Full Only' in instruction_structure:
        for ex in chosen_examples:
            generic_instruction += "\ninput: " + ex['input'] + "\noutput: " + ex['output']

    promptlist, answerlist = [], []
    for i in test_indices:
        if null_word is None:
            prompt = generic_instruction + "\n" + instances[i]['input'] + "\nSummary:"
        else:
            prompt = generic_instruction + "\n" + null_word + "\nSummary:"
        promptlist.append([
            {"role": "system", "content": "You are an expert clinical summarizer."},
            {"role": "user",   "content": prompt}
        ])
        answerlist.append(
            instances[i]['output'][0] if isinstance(instances[i]['output'], list)
            else instances[i]['output']
        )
    return promptlist, answerlist, test_indices


def training_encode_instruction(instruction_structure=['Definition'], number_of_examples=0,
                                number_of_instances=100, null_word=None, data_seed=0,
                                modified={}, args=args):
    random.seed(0)
    instances = DATASET['Instances']
    instance_indices = list(range(len(instances)))
    test_indices = random.sample(instance_indices, min(number_of_instances, len(instances)))
    remaining_indices = [i for i in instance_indices if i not in test_indices]

    random.seed(data_seed)
    pos_examples = DATASET['Positive Examples']
    chosen_examples = random.sample(pos_examples, min(number_of_examples, len(pos_examples))) \
                      if number_of_examples > 0 else []

    definition = modified.get('Definition', DATASET['Definition'])
    generic_instruction = 'Definition: ' + definition.strip()
    if 'Positive Examples Full Only' in instruction_structure:
        for ex in chosen_examples:
            generic_instruction += "\ninput: " + ex['input'] + "\noutput: " + ex['output']

    promptlist, answerlist = [], []
    for i in test_indices:
        prompt = generic_instruction + "\n" + instances[i]['input'] + "\nSummary:"
        promptlist.append([
            {"role": "system", "content": "You are an expert clinical summarizer."},
            {"role": "user",   "content": prompt}
        ])
        answerlist.append(
            instances[i]['output'][0] if isinstance(instances[i]['output'], list)
            else instances[i]['output']
        )

    train_promptlist, train_answerlist, train_indexlist = [], [], []
    train_indices = random.sample(remaining_indices, min(args.num_train, len(remaining_indices)))
    for i in train_indices:
        prompt = generic_instruction + "\n" + instances[i]['input'] + "\nSummary:"
        train_promptlist.append([
            {"role": "system", "content": "You are an expert clinical summarizer."},
            {"role": "user",   "content": prompt}
        ])
        train_answerlist.append(
            instances[i]['output'][0] if isinstance(instances[i]['output'], list)
            else instances[i]['output']
        )
        train_indexlist.append(i)

    return (promptlist, answerlist, test_indices,
            train_promptlist, train_answerlist, train_indexlist,
            [], [], [])


def create_batches(test_instances, test_labels=[], batch_size=4):
    test_sentence_batches, test_label_batches = [], []
    for i in range(0, len(test_instances), batch_size):
        test_sentence_batches.append(test_instances[i:i+batch_size])
        if len(test_labels) > 0:
            test_label_batches.append(test_labels[i:i+batch_size])
    return (test_sentence_batches, test_label_batches) if test_labels else test_sentence_batches


def construct_instruction_prompt(mode, num_shots, num_test_instances, data_seed,
                                  null_word=None, args=args):
    if mode == "Instruction Only":
        return encode_instruction(
            instruction_structure=["Definition"],
            number_of_examples=num_shots,
            number_of_instances=num_test_instances,
            data_seed=data_seed,
            null_word=null_word,
            args=args
        )
    else:
        raise ValueError("Invalid mode")


def custom_instruction_prompt(mode=args.mode, task_name=None, num_shots=args.num_shots,
                               num_test_instances=100, data_seed=None, null_word=None,
                               split='train', modified={}, args=args):
    if mode == "Instruction Only":
        result = training_encode_instruction(
            instruction_structure=["Definition"],
            number_of_examples=num_shots,
            number_of_instances=num_test_instances,
            data_seed=data_seed,
            null_word=null_word,
            modified=modified,
            args=args
        )
    else:
        raise ValueError()
    if split == 'test':
        return result[:3]
    elif split == 'train':
        (prompt_list, answer_list, index_list,
         train_prompt_list, train_answer_list, train_index_list,
         dev_prompt_list, dev_answerlist, dev_index_list) = result
        train_prompt_list.extend(dev_prompt_list)
        train_answer_list.extend(dev_answerlist)
        train_index_list.extend(dev_index_list)
        try:
            random.seed(data_seed)
            indices = random.sample(range(len(train_index_list)), args.num_train)
            train_prompt_list = [train_prompt_list[i] for i in indices]
            train_answer_list = [train_answer_list[i] for i in indices]
            train_index_list  = [train_index_list[i]  for i in indices]
        except Exception as e:
            tqdm.write(f"Error in sampling: {e}")
        return train_prompt_list, train_answer_list, train_index_list
    else:
        raise ValueError()


## Cell 7: Run + Score Functions

In [ ]:
def run(mode, batch_size, num_shots, num_samples, data_seed=0,
        override_prompts=False, function=None, split=None, modified={}, args=args):
    if not override_prompts:
        prompt_list, answer_list, index_list = construct_instruction_prompt(
            mode=mode, num_shots=num_shots,
            num_test_instances=num_samples, data_seed=data_seed, args=args
        )
    else:
        prompt_list, answer_list, index_list = function(
            mode=mode, num_shots=num_shots,
            num_test_instances=num_samples, data_seed=data_seed,
            null_word=None, split=split, modified=modified, args=args
        )

    prompt_batches, batch_test_labels = create_batches(prompt_list, answer_list, batch_size)
    predictions = []
    for batch in prompt_batches:
        for prompt in batch:
            pred = complete_phi4(prompt)
            predictions.append(pred)

    rouge_scorer_obj = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_scores = [rouge_scorer_obj.score(ref, pred)['rougeL'].fmeasure
                    for ref, pred in zip(answer_list, predictions)]
    bert_scores = bert_score(predictions, answer_list, lang="en", verbose=False)[2].tolist()

    smoothie = SmoothingFunction().method4
    bleu_scores = [
        sentence_bleu([word_tokenize(ref.lower())], word_tokenize(pred.lower()),
                      smoothing_function=smoothie)
        for pred, ref in zip(predictions, answer_list)
    ]

    avg_rouge = np.mean(rouge_scores) * 100
    avg_bert  = np.mean(bert_scores)  * 100
    avg_bleu  = np.mean(bleu_scores)  * 100

    return predictions, avg_rouge, avg_bert, avg_bleu, answer_list, index_list


def score(candidate, split='test', write=False):
    predictions, avg_rouge, avg_bert, avg_bleu, answer_list, indexlist = run(
        mode=args.mode, batch_size=args.batch_size, num_shots=args.num_shots,
        num_samples=100, data_seed=args.seed,
        override_prompts=True, function=custom_instruction_prompt,
        split=split, modified={'Definition': candidate}, args=args
    )
    summarization_score = 0.6 * avg_rouge + 0.4 * avg_bert
    if split == 'test' and write:
        pname = args.meta_name.split('.')[0] + "_predictions.json"
        pred_dump = {'predictions': predictions, 'answers': answer_list, 'ids': indexlist}
        with open(os.path.join(args.meta_dir, pname), 'w+') as pfile:
            json.dump(pred_dump, pfile)
    return summarization_score


def evaluate_objectives(candidate, split='train'):
    if candidate in evaluation_cache:
        return evaluation_cache[candidate]

    predictions, avg_rouge, avg_bert, avg_bleu, answer_list, indexlist = run(
        mode=args.mode, batch_size=args.batch_size, num_shots=args.num_shots,
        num_samples=100, data_seed=args.seed,
        override_prompts=True, function=custom_instruction_prompt,
        split=split, modified={'Definition': candidate}, args=args
    )
    summarization_score = 0.6 * avg_rouge + 0.4 * avg_bert
    grammar_score = get_llm_grammar_score(candidate)

    with open(os.path.join(args.meta_dir, 'rouge_scores.txt'), 'a') as f:
        f.write(f"Candidate: {candidate}\nAverage ROUGE Score: {avg_rouge:.4f}\n\n")
    with open(os.path.join(args.meta_dir, 'bert_scores.txt'), 'a') as f:
        f.write(f"Candidate: {candidate}\nAverage BERT Score: {avg_bert:.4f}\n\n")
    with open(os.path.join(args.meta_dir, 'bleu_scores.txt'), 'a') as f:
        f.write(f"Candidate: {candidate}\nAverage BLEU Score: {avg_bleu:.4f}\n\n")

    evaluation_cache[candidate] = (summarization_score, grammar_score, avg_rouge, avg_bert, avg_bleu)
    return summarization_score, grammar_score, avg_rouge, avg_bert, avg_bleu


## Cell 8: Grammar + NLP Edit Utility Functions

In [ ]:
from supar import Parser
parser_crf = Parser.load('crf-con-en')


def detokenize(tokens):
    return TreebankWordDetokenizer().detokenize(tokens)


def containenglish(text):
    return bool(re.search('[a-zA-Z]', text))


def check_child(tree):
    count, total_count = 0, 0
    for subtree in tree:
        total_count += 1
        if isinstance(subtree, nltk.tree.Tree) and subtree.label() == '_':
            count += 1
    return count >= total_count - count


def collect_leaves(parsed_tree):
    leaves = []
    for tree in parsed_tree:
        if not isinstance(tree, nltk.tree.Tree):
            continue
        if tree.label() == '_':
            leaves.append(detokenize(tree.leaves()))
            continue
        if check_child(tree):
            leaves.append(detokenize(tree.leaves()))
        else:
            leaves.extend(collect_leaves(tree))
    return leaves


def get_phrases(instruction):
    phrases = []
    for sentence in sent_tokenize(instruction):
        parsed_tree = parser_crf.predict(word_tokenize(sentence), verbose=False).sentences[0].trees[0]
        leaves = collect_leaves(parsed_tree)
        phrases.extend(leaves)
    exclude_words = {'he', 'she', 'it', 'they', 'i', 'we', 'me', 'him', 'her', 'them', 'us'}
    normalized_phrases = []
    for phrase in phrases:
        if phrase not in string.punctuation and phrase != '':
            norm = re.sub(r'\s+', ' ', phrase.strip())
            norm = re.sub(r',\s*', ', ', norm)
            norm = re.sub(r"(')(\S+)(\s+)(')", r"\1\2\4", norm)
            tokens = word_tokenize(norm.lower())
            if len(tokens) == 1 and tokens[0] in exclude_words:
                continue
            normalized_phrases.append(norm)
    return normalized_phrases


def get_phrase_lookup_pun(base_candidate):
    phrases = []
    for sentence in sent_tokenize(base_candidate):
        parsed_tree = parser_crf.predict(word_tokenize(sentence), verbose=False).sentences[0].trees[0]
        leaves = collect_leaves(parsed_tree)
        phrases.extend(leaves)
    phrases = [detokenize(word_tokenize(phrase)) for phrase in phrases
               if phrase.strip() and not all(c in string.punctuation for c in phrase.strip())]
    phrase_lookup = {p: phrase for p, phrase in enumerate(phrases)}
    tqdm.write(f"Extracted phrases (punctuation excluded): {phrases}")
    meta_file.write(f"Extracted phrases (punctuation excluded): {str(phrases)}\n")
    return phrase_lookup


def get_llm_grammar_score(text):
    system_prompt = (
        "You are a strict grammar evaluator. Score grammar from 0 to 100. "
        "100 = perfect grammar. 0 = very poor grammar. Return only a number, no text."
    )
    user_prompt = (
        "Evaluate the grammar of this text and return a score from 0 to 100.\n"
        "Text:\n\"\"\"\n" + text + "\n\"\"\""
    )
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt}
    ]
    try:
        raw_output = complete_phi4(messages, max_tokens=5)
        tqdm.write(f"Raw grammar output for '{text}': '{raw_output}'")
        match = re.match(r'^\[?(\d+)\]?$', raw_output.strip())
        if match:
            sc = int(match.group(1))
        else:
            numbers = re.findall(r'\d+', raw_output)
            sc = int(numbers[0]) if numbers else None
            if sc is None:
                raise ValueError("No valid number found")
        return sc if 0 <= sc <= 100 else language_tool_fallback(text)
    except (ValueError, TypeError) as e:
        tqdm.write(f"Failed to parse grammar output, using LanguageTool fallback: {e}")
        return language_tool_fallback(text)
    except RuntimeError as e:
        if "CUDA out of memory" in str(e):
            torch.cuda.empty_cache(); gc.collect()
            return language_tool_fallback(text)
        raise e


def language_tool_fallback(text):
    matches = tool.check(text)
    sc = 100
    for match in matches:
        if match.ruleId.startswith('MORFOLOGIK_') or match.ruleId == 'UPPERCASE_SENTENCE_START':
            sc -= 5
        elif 'AGREEMENT' in match.ruleId:
            sc -= 15
        elif 'GRAMMAR' in match.ruleId or 'SENTENCE' in match.ruleId:
            sc -= 20
        else:
            sc -= 10
    return max(0, sc)


def substitute_phrase(candidate, phrase):
    system_prompt = "You are a grammar and paraphrasing expert. Your task is to paraphrase a phrase so it fits grammatically and contextually within an instruction."
    user_prompt = (
        "Given a text and a phrase, provide the best paraphrase for that phrase which fits perfectly in the text.\n"
        "Instructional text: " + candidate + "\n"
        "Phrase to paraphrase: " + phrase + "\n"
        "Only return the paraphrased phrase—no extra text or explanation.\n"
        "Paraphrased phrase:"
    )
    messages = [{"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt}]
    try:
        paraphrase = complete_phi4(messages, max_tokens=30).strip('.')
        paraphrase = paraphrase.strip('\'\'"')
        paraphrase = re.sub(r'^(Paraphrased phrase:)\s*', '', paraphrase)
        if not paraphrase.strip():
            tqdm.write("Empty paraphrase generated, returning original phrase")
            paraphrase = phrase
        tqdm.write("Substituting phrase: " + phrase + " with: " + paraphrase)
        if candidate.find(' ' + phrase + ' ') > 0:
            full_prompt = candidate.replace(' ' + phrase + ' ', ' ' + paraphrase + ' ')
        elif candidate.find(phrase + ' ') > 0:
            full_prompt = candidate.replace(phrase + ' ', paraphrase + ' ')
        elif candidate.find(' ' + phrase) > 0:
            full_prompt = candidate.replace(' ' + phrase, ' ' + paraphrase)
        else:
            full_prompt = candidate.replace(phrase, paraphrase)
        if get_llm_grammar_score(full_prompt) < 10:
            tqdm.write(f"Low grammar after substitution, reverting")
            paraphrase = phrase
    except Exception as e:
        tqdm.write(f"Error during paraphrasing: {e}, returning original phrase")
        paraphrase = phrase
    if candidate.find(' ' + phrase + ' ') > 0:
        return candidate.replace(' ' + phrase + ' ', ' ' + paraphrase + ' ')
    elif candidate.find(phrase + ' ') > 0:
        return candidate.replace(phrase + ' ', paraphrase + ' ')
    elif candidate.find(' ' + phrase) > 0:
        return candidate.replace(' ' + phrase, ' ' + paraphrase)
    return candidate.replace(phrase, paraphrase)


def delete_phrase(candidate, phrase):
    if candidate.find(' ' + phrase) > 0:
        return candidate.replace(' ' + phrase, ' ')
    elif candidate.find(phrase + ' ') > 0:
        return candidate.replace(phrase + ' ', ' ')
    return candidate.replace(phrase, '')


def add_phrase(candidate, phrase, after):
    if after == '':
        return phrase + ' ' + candidate
    if candidate.find(' ' + after) > 0:
        return candidate.replace(' ' + after, ' ' + after + ' ' + phrase)
    elif candidate.find(after + ' ') > 0:
        return candidate.replace(after + ' ', after + ' ' + phrase + ' ')
    return candidate.replace(after, after + phrase)


def swap_phrases(candidate, phrase_1, phrase_2):
    if phrase_1 in phrase_2:
        candidate = candidate.replace(' ' + phrase_2 + ' ', ' <2> ') \
                    if candidate.find(' ' + phrase_2 + ' ') >= 0 \
                    else candidate.replace(phrase_2, '<2>')
        answer = candidate
        answer = answer.replace(' ' + phrase_1 + ' ', ' <1> ') \
                 if answer.find(' ' + phrase_1 + ' ') >= 0 \
                 else answer.replace(phrase_1, '<1>')
    else:
        candidate = candidate.replace(' ' + phrase_1 + ' ', ' <1> ') \
                    if candidate.find(' ' + phrase_1 + ' ') >= 0 \
                    else candidate.replace(phrase_1, '<1>')
        answer = candidate
        answer = answer.replace(' ' + phrase_2 + ' ', ' <2> ') \
                 if answer.find(' ' + phrase_2 + ' ') >= 0 \
                 else answer.replace(phrase_2, '<2>')
    answer = answer.replace('<1>', phrase_2).replace('<2>', phrase_1)
    return answer


def perform_edit(edit, base_text, phrase_list, deleted_phrases_history):
    mutated = base_text
    if edit == 'del':
        if not phrase_list:
            tqdm.write("No matching phrase found for deletion.")
            return base_text, deleted_phrases_history
        chosen = random.choice(phrase_list)
        phrase_list.remove(chosen)
        tqdm.write("Deleting phrase: " + chosen)
        mutated = delete_phrase(base_text, chosen)
        deleted_phrases_history.append(chosen)
    elif edit == 'swap':
        if len(phrase_list) < 2:
            tqdm.write("Not enough matching phrases found for swap.")
            return base_text, deleted_phrases_history
        p1, p2 = random.sample(phrase_list, 2)
        for p in (p1, p2):
            if p in phrase_list: phrase_list.remove(p)
        tqdm.write("Swapping phrases: " + p1 + " and " + p2)
        mutated = swap_phrases(base_text, p1, p2)
    elif edit == 'sub':
        if not phrase_list:
            tqdm.write("No matching phrase found for substitution.")
            return base_text, deleted_phrases_history
        chosen = random.choice(phrase_list)
        phrase_list.remove(chosen)
        tqdm.write("Substituting phrase: " + chosen)
        mutated = substitute_phrase(base_text, chosen)
    elif edit == 'add':
        if not deleted_phrases_history:
            tqdm.write("No deleted phrases available for addition.")
            return base_text, deleted_phrases_history
        phrase_to_add = random.choice(deleted_phrases_history)
        if phrase_list:
            after = random.choice(phrase_list)
            if after in phrase_list: phrase_list.remove(after)
            tqdm.write("Adding phrase: " + phrase_to_add + " after " + after)
            mutated = add_phrase(base_text, phrase_to_add, after)
        else:
            tqdm.write("No matching phrase found for 'add'; prepending: " + phrase_to_add)
            mutated = phrase_to_add + " " + base_text
    return mutated, deleted_phrases_history


def safe_mutation(edit, base_text, phrase_list, deleted_phrases_history,
                  grammar_threshold=50, similarity_threshold=0.0):
    mutated, new_del = perform_edit(edit, base_text, phrase_list, deleted_phrases_history)
    gscore = get_llm_grammar_score(mutated)
    if gscore >= grammar_threshold:
        summarization_score, _, _, _, _ = evaluate_objectives(mutated)
        tqdm.write(f"After {edit}: grammar={gscore}, summarization={summarization_score}")
    else:
        tqdm.write(f"After {edit}: grammar={gscore} — rejected (below threshold)")
        return base_text, deleted_phrases_history
    return mutated, new_del


## Cell 9: GA Operators — Tournament, Crossover, Pareto Plot

In [ ]:
def tournament_selection(population, population_scores, num_tournaments):
    S_candidates, S_scores = [], []
    for _ in range(num_tournaments):
        parent = np.random.randint(0, len(population))
        S_candidates.append(population[parent])
        S_scores.append(population_scores[parent])
    base_idx = np.argmax(S_scores)
    return S_candidates[base_idx], S_scores[base_idx]


def crossover(parent_1, parent_2):
    flag_error = False
    max_attempts = 3
    attempt = 0
    while attempt < max_attempts:
        try:
            phrases_1_pun = get_phrase_lookup_pun(parent_1)
            phrases_2_pun = get_phrase_lookup_pun(parent_2)
        except AttributeError:
            tqdm.write("AttributeError during phrase extraction in crossover")
            meta_file.write("AttributeError during phrase extraction in crossover\n")
            return parent_1, True

        phrases_1 = list(phrases_1_pun.values())
        phrases_2 = list(phrases_2_pun.values())

        if not phrases_1 or not phrases_2:
            tqdm.write("No valid phrases for crossover")
            meta_file.write("No valid phrases for crossover\n")
            return parent_1, True

        total_phrases = min(len(phrases_1), len(phrases_2))
        num_from_p1 = random.randint(1, total_phrases - 1) if total_phrases > 1 else 1
        num_from_p2 = total_phrases - num_from_p1
        random.shuffle(phrases_1)
        random.shuffle(phrases_2)
        offspring_phrases = phrases_1[:num_from_p1] + phrases_2[:num_from_p2]
        offspring_words = [w for phrase in offspring_phrases for w in word_tokenize(phrase)]
        offspring = detokenize(offspring_words)

        grammar_score = get_llm_grammar_score(offspring)
        if containenglish(offspring) and grammar_score >= 50:
            tqdm.write(f"Generated offspring (attempt {attempt+1}): {offspring}, Grammar: {grammar_score}")
            meta_file.write(f"Generated offspring (attempt {attempt+1}): {offspring}, Grammar: {grammar_score}\n")
            if 'wandb' in globals():
                wandb.log({"Crossover Offspring": offspring, "Grammar Score": grammar_score})
            return offspring, flag_error
        else:
            tqdm.write(f"Offspring rejected (attempt {attempt+1}): Grammar={grammar_score}")
            meta_file.write(f"Offspring rejected (attempt {attempt+1}): Grammar={grammar_score}\n")
            attempt += 1

    tqdm.write("All crossover attempts failed, returning parent_1")
    meta_file.write("All crossover attempts failed, returning parent_1\n")
    if 'wandb' in globals():
        wandb.log({"Crossover Failed": "All attempts exhausted"})
    return parent_1, True


def plot_pareto_front(population_data, fronts, iteration, meta_dir, final=False):
    plt.figure(figsize=(8, 6))
    summarization_scores = [d[1] for d in population_data]
    grammar_scores       = [d[2] for d in population_data]
    plt.scatter(summarization_scores, grammar_scores, c='gray', alpha=0.5, label='All Candidates')
    colors = ['red', 'blue', 'green', 'orange', 'purple']
    for front_idx, front in enumerate(fronts):
        if front_idx >= len(colors): break
        fs = [population_data[i][1] for i in front]
        fg = [population_data[i][2] for i in front]
        sorted_idx = np.argsort(fs)
        label = 'Pareto Front' if front_idx == 0 else f'Front {front_idx+1}'
        plt.scatter(fs, fg, c=colors[front_idx], label=label)
        plt.plot([fs[i] for i in sorted_idx], [fg[i] for i in sorted_idx],
                 c=colors[front_idx], linestyle='--')
    plt.xlabel('Summarization Score'); plt.ylabel('Grammar Score')
    title = 'Pareto Optimal Curve (Final)' if final else f'Pareto Optimal Curve (Iteration {iteration})'
    plt.title(title); plt.legend(); plt.grid(True)
    plot_name = 'pareto_final.png' if final else f'pareto_iteration_{iteration}.png'
    plot_path = os.path.join(meta_dir, plot_name)
    plt.savefig(plot_path); plt.close()
    if 'wandb' in globals():
        wandb.log({title: wandb.Image(plot_path)})
    return plot_path


## Cell 10: Initial Population — 10 Clinical Prompts + Fresh Scoring

In [ ]:
initial_candidates = [
    # Original prompt
    ('Write a concise clinical summary of the following case report. '
     'Keep all the below points in mind: '
     '1. Patient demographics (age, sex) '
     '2. Presenting complaint and relevant medical history '
     '3. Key clinical findings and investigations '
     '4. Diagnosis '
     '5. Treatment plan '
     '6. Clinical outcome '
     'Use clear, professional medical language.'),
    # Paraphrase 1
    ('Summarize the following clinical case report concisely. '
     'Ensure your summary covers: '
     '1. Patient demographics (age, sex) '
     '2. Chief complaint and pertinent medical history '
     '3. Significant clinical findings and diagnostic results '
     '4. Diagnosis '
     '5. Management plan '
     '6. Patient outcome '
     'Use precise, professional medical terminology.'),
    # Paraphrase 2
    ('Provide a brief clinical summary of the case report below. '
     'Your summary must address: '
     '1. Patient demographics (age, sex) '
     '2. Primary complaint and relevant history '
     '3. Notable clinical findings and investigations '
     '4. Diagnosis '
     '5. Treatment approach '
     '6. Clinical outcome '
     'Write in formal medical language.'),
    # Paraphrase 3
    ('Generate a concise summary for the clinical case report provided. '
     'Include the following: '
     '1. Patient demographics (age, sex) '
     '2. Presenting symptoms and medical background '
     '3. Key examination findings and test results '
     '4. Diagnosis '
     '5. Therapeutic plan '
     '6. Outcome of treatment '
     'Maintain clear and professional clinical language.'),
    # Paraphrase 4
    ('Write a structured clinical summary of the case report given below. '
     'Cover these aspects: '
     '1. Patient demographics (age, sex) '
     '2. Main complaint and prior medical history '
     '3. Critical clinical observations and investigations '
     '4. Diagnosis '
     '5. Treatment strategy '
     '6. Final clinical outcome '
     'Use appropriate medical terminology throughout.'),
    # Paraphrase 5
    ('Compose a brief summary of the following medical case report. '
     'Address the points below: '
     '1. Demographics of the patient (age, sex) '
     '2. Presenting complaint and clinical history '
     '3. Important findings and investigative results '
     '4. Diagnosis '
     '5. Course of treatment '
     "6. Patient's clinical outcome "
     'Employ professional and concise medical language.'),
    # Paraphrase 6
    ('Produce a concise clinical overview of the case report below. '
     'Your response should include: '
     '1. Patient demographics (age, sex) '
     '2. Reason for presentation and relevant medical history '
     '3. Key clinical and investigative findings '
     '4. Diagnosis '
     '5. Treatment plan '
     '6. Outcome '
     'Use formal medical language throughout.'),
    # Paraphrase 7
    ('Summarize the medical case report below in a clinical and professional manner. '
     'Make sure to cover: '
     '1. Patient demographics (age, sex) '
     '2. Complaint at presentation and relevant history '
     '3. Significant findings from examination and investigations '
     '4. Diagnosis '
     '5. Recommended treatment '
     '6. Clinical result '
     'Keep the language precise and medically appropriate.'),
    # Paraphrase 8
    ('Draft a brief and structured summary of the clinical case report provided. '
     'The summary should capture: '
     '1. Patient demographics (age, sex) '
     '2. Presenting issue and medical history '
     '3. Relevant clinical findings and diagnostic workup '
     '4. Diagnosis '
     '5. Treatment administered '
     '6. Outcome observed '
     'Use clear, concise medical language.'),
    # Paraphrase 9
    ('Formulate a concise clinical summary based on the case report below. '
     'Incorporate the following elements: '
     '1. Patient demographics (age, sex) '
     '2. Initial complaint and pertinent history '
     '3. Key findings from clinical assessment and investigations '
     '4. Diagnosis '
     '5. Treatment plan implemented '
     '6. Clinical prognosis and outcome '
     'Write using professional medical language.'),
]

# Adjust population_size to match candidate count
args.population_size = len(initial_candidates)
population_size      = args.population_size
tqdm.write(f"Population size set to {population_size}")
meta_file.write(f"Population size: {population_size}\n")

# Score initial candidates fresh
tqdm.write("Scoring initial population...")
initial_scores = []
for cand in initial_candidates:
    s_score, g_score, r_score, b_score, bl_score = evaluate_objectives(cand)
    weighted = 0.6 * s_score + 0.4 * g_score
    initial_scores.append({
        "candidate":           cand,
        "summarization_score": s_score,
        "grammar_score":       g_score,
        "rouge_score":         r_score,
        "bert_score":          b_score,
        "bleu_score":          bl_score,
        "weighted_score":      weighted
    })
    tqdm.write(f"Scored: {cand[:60]}... | summ={s_score:.2f}, gram={g_score:.2f}")
    meta_file.write(f"Initial Candidate: {cand}\nScores: summ={s_score:.4f}, gram={g_score:.4f}, "
                    f"rouge={r_score:.4f}, bert={b_score:.4f}, bleu={bl_score:.4f}, weighted={weighted:.4f}\n")
    if 'wandb' in globals():
        wandb.log({"Initial Candidate": cand, "Initial Summarization Score": s_score,
                   "Initial Grammar Score": g_score, "Initial ROUGE": r_score,
                   "Initial BERT": b_score, "Initial BLEU": bl_score})

best_initial             = max(initial_scores, key=lambda x: x["weighted_score"])
best_candidate           = best_initial["candidate"]
best_summarization_score = best_initial["summarization_score"]
best_grammar_score       = best_initial["grammar_score"]
best_rouge_score         = best_initial["rouge_score"]
best_bert_score          = best_initial["bert_score"]
best_bleu_score          = best_initial["bleu_score"]

tqdm.write(f"Best Initial Candidate: {best_candidate}")
tqdm.write(f"Best Initial Scores (Summarization, Grammar, ROUGE, BERT, BLEU): "
          f"({best_summarization_score:.4f}, {best_grammar_score:.4f}, {best_rouge_score:.4f}, {best_bert_score:.4f}, {best_bleu_score:.4f})")
meta_file.write(f"Best Initial Candidate: {best_candidate}\n")
meta_file.write(f"Best Initial Scores: summ={best_summarization_score:.4f}, gram={best_grammar_score:.4f}, "
                f"rouge={best_rouge_score:.4f}, bert={best_bert_score:.4f}, bleu={best_bleu_score:.4f}\n")
if 'wandb' in globals():
    wandb.log({"Best Initial Candidate": best_candidate,
               "Best Initial Summarization Score": best_summarization_score,
               "Best Initial Grammar Score": best_grammar_score,
               "Best Initial ROUGE Score": best_rouge_score,
               "Best Initial BERT Score": best_bert_score,
               "Best Initial BLEU Score": best_bleu_score})

best_rouge_scores         = [best_rouge_score]
best_bert_scores          = [best_bert_score]
best_bleu_scores          = [best_bleu_score]
best_summarization_scores = [best_summarization_score]
best_grammar_scores       = [best_grammar_score]

if len(initial_candidates) <= args.population_size:
    W_candidates = initial_candidates
    W_objectives = [(s["summarization_score"], s["grammar_score"]) for s in initial_scores]
    W_deletesets = [[] for _ in range(len(W_candidates))]
else:
    sorted_scores = sorted(initial_scores, key=lambda x: x["weighted_score"], reverse=True)[:args.population_size]
    W_candidates  = [s["candidate"] for s in sorted_scores]
    W_objectives  = [(s["summarization_score"], s["grammar_score"]) for s in sorted_scores]
    W_deletesets  = [[] for _ in range(len(W_candidates))]
    tqdm.write(f"Selected top {args.population_size} initial candidates based on weighted score.")
    meta_file.write(f"Selected top {args.population_size} initial candidates based on weighted score.\n")

tqdm.write("Initial Population:")
for candidate, obj in zip(W_candidates, W_objectives):
    tqdm.write(str({"prompt": candidate, "summarization_score": obj[0], "grammar_score": obj[1]}))
if 'wandb' in globals():
    wandb.log({"Initial Population": W_objectives})

plot_pareto_front.best_candidate   = best_candidate
plot_pareto_front.patience_counter = 0

WEIGHT_SUMM = 0.6
WEIGHT_GRAM = 0.4


## Cell 11: Main MOGA Evolutionary Loop

In [ ]:
def non_dominated_sorting(population):
    n = len(population)
    domination_count = [0] * n
    dominated_set    = [[] for _ in range(n)]
    fronts = []
    for i in range(n):
        for j in range(n):
            if i == j: continue
            p_s, p_g = population[i][1], population[i][2]
            q_s, q_g = population[j][1], population[j][2]
            if (p_s >= q_s and p_g >= q_g) and (p_s > q_s or p_g > q_g):
                dominated_set[i].append(j)
            elif (q_s >= p_s and q_g >= p_g) and (q_s > p_s or q_g > p_g):
                domination_count[i] += 1
    front0 = [i for i in range(n) if domination_count[i] == 0]
    fronts.append(front0)
    current_front = front0
    while current_front:
        next_front = []
        for i in current_front:
            for j in dominated_set[i]:
                domination_count[j] -= 1
                if domination_count[j] == 0:
                    next_front.append(j)
        if next_front: fronts.append(next_front)
        current_front = next_front
    return fronts


def compute_crowding_distance(population_data, front):
    distances = {i: 0.0 for i in front}
    for obj_idx in [1, 2]:
        sorted_front = sorted(front, key=lambda i: population_data[i][obj_idx])
        obj_min = population_data[sorted_front[0]][obj_idx]
        obj_max = population_data[sorted_front[-1]][obj_idx]
        distances[sorted_front[0]]  = float('inf')
        distances[sorted_front[-1]] = float('inf')
        for k in range(1, len(sorted_front) - 1):
            norm_diff = 0.0 if obj_max - obj_min == 0 else \
                (population_data[sorted_front[k+1]][obj_idx] -
                 population_data[sorted_front[k-1]][obj_idx]) / (obj_max - obj_min)
            distances[sorted_front[k]] += norm_diff
    return distances


start_time = time.time()

for iter_idx in range(num_steps):
    tqdm.write(f"\n----- Iteration: {iter_idx} -----")
    meta_file.write(f"Running step:\t{iter_idx}\n")

    tqdm.write("Current Population:")
    for cand, obj in zip(W_candidates, W_objectives):
        tqdm.write(str({"prompt": cand, "summarization_score": obj[0], "grammar_score": obj[1]}))
    if 'wandb' in globals():
        wandb.log({f"Current Population iter {iter_idx}": W_objectives})

    new_candidates = W_candidates.copy()
    new_objectives = W_objectives.copy()
    new_deletesets = W_deletesets.copy()

    # Crossover
    for j in range(num_offspring):
        parent_1, _ = tournament_selection(W_candidates, [obj[0] for obj in W_objectives], num_tournaments)
        parent_2, _ = tournament_selection(W_candidates, [obj[0] for obj in W_objectives], num_tournaments)
        meta_file.write("Parent 1 (" + str(j) + "):\t " + parent_1 + '\n')
        meta_file.write("Parent 2 (" + str(j) + "):\t " + parent_2 + '\n')
        tqdm.write("Parent 1 (" + str(j) + "): " + parent_1)
        tqdm.write("Parent 2 (" + str(j) + "): " + parent_2)
        offspring, flag_error = crossover(parent_1, parent_2)
        if flag_error or not containenglish(offspring):
            meta_file.write("Crossover skipped due to error or non-English offspring\n")
            tqdm.write("Crossover skipped due to error or non-English offspring")
            if 'wandb' in globals():
                wandb.log({"Crossover Skipped": f"Iteration {iter_idx}, Offspring {j}"})
            continue
        meta_file.write("Offspring (" + str(j) + "):\t " + offspring + '\n')
        tqdm.write("Offspring (" + str(j) + "): " + offspring)
        new_candidates.append(offspring)
        new_deletesets.append(new_deletesets[W_candidates.index(parent_1)])

    # Mutation
    for idx, base_candidate in enumerate(new_candidates[:population_size]):
        if mutation_prob > np.random.random():
            try:
                phrase_list = get_phrases(base_candidate)
                tqdm.write("Initial phrases for candidate mutation: " + str(phrase_list))
            except AttributeError:
                tqdm.write("Mutation skipped due to error"); continue
            if use_add and not new_deletesets[idx]:
                if 'add' in edit_operations: edit_operations.remove('add')
            edits = np.random.choice(edit_operations, num_candidates)
            tqdm.write("Sampled edit operations for mutation: " + str(edits))
            grammar_threshold    = 85
            similarity_threshold = 0.0
            for edit_op in edits:
                mutated, new_deletesets[idx] = safe_mutation(
                    edit_op, base_candidate, phrase_list.copy(), new_deletesets[idx],
                    grammar_threshold=grammar_threshold, similarity_threshold=similarity_threshold
                )
                if mutated != base_candidate:
                    new_candidates[idx] = mutated
                    break

    # Evaluate
    new_objectives   = []
    candidate_scores = []
    for candidate in new_candidates:
        s, g, r, b, bl = evaluate_objectives(candidate)
        new_objectives.append((s, g))
        candidate_scores.append((r, b, bl, s, g))

    combined        = list(zip(new_candidates, new_objectives, new_deletesets))
    population_data = [(c, o[0], o[1], d) for c, o, d in combined]

    fronts = non_dominated_sorting(population_data)
    tqdm.write(f"Non-dominated fronts (by candidate indices): {fronts}")
    meta_file.write(f"Non-dominated fronts (by candidate indices): {str(fronts)}\n")

    plot_pareto_front(population_data, fronts, iter_idx, args.meta_dir)

    final_population_indices = []
    remaining = population_size
    for front in fronts:
        if len(front) <= remaining:
            final_population_indices.extend(front)
            remaining -= len(front)
        else:
            distances    = compute_crowding_distance(population_data, front)
            sorted_front = sorted(front, key=lambda i: distances[i], reverse=True)
            final_population_indices.extend(sorted_front[:remaining])
            remaining = 0
        if remaining == 0: break

    W_candidates = [population_data[i][0] for i in final_population_indices]
    W_objectives = [(population_data[i][1], population_data[i][2]) for i in final_population_indices]
    W_deletesets = [population_data[i][3] for i in final_population_indices]

    tqdm.write("Updated Population at end of iteration {}:".format(iter_idx))
    for candidate, obj in zip(W_candidates, W_objectives):
        tqdm.write(str({"prompt": candidate, "summarization_score": obj[0], "grammar_score": obj[1]}))

    pareto_front = fronts[0]
    best_idx = pareto_front[0] if len(pareto_front) == 1 else max(
        pareto_front,
        key=lambda i: WEIGHT_SUMM * population_data[i][1] + WEIGHT_GRAM * population_data[i][2]
    )
    result_candidate  = population_data[best_idx][0]
    result_objectives = (population_data[best_idx][1], population_data[best_idx][2])

    best_scores = candidate_scores[best_idx]
    best_rouge_scores.append(best_scores[0])
    best_bert_scores.append(best_scores[1])
    best_bleu_scores.append(best_scores[2])
    best_summarization_scores.append(best_scores[3])
    best_grammar_scores.append(best_scores[4])

    tqdm.write("Best Candidate this iteration: " + result_candidate)
    tqdm.write("Best Candidate Objectives (Summarization, Grammar): " + str(result_objectives))
    tqdm.write(f"Best Candidate Scores (ROUGE, BERT, BLEU, Summarization, Grammar): {best_scores}")
    if 'wandb' in globals():
        wandb.log({"Best Candidate": result_candidate,
                   "Best Candidate Objectives": result_objectives,
                   "Best ROUGE Score": best_scores[0],
                   "Best BERT Score":  best_scores[1],
                   "Best BLEU Score":  best_scores[2],
                   "Best Summarization Score": best_scores[3],
                   "Best Grammar Score": best_scores[4]})

    if not hasattr(plot_pareto_front, 'best_candidate'):
        plot_pareto_front.best_candidate   = result_candidate
        plot_pareto_front.patience_counter = 0
    else:
        if result_candidate == plot_pareto_front.best_candidate:
            plot_pareto_front.patience_counter += 1
        else:
            plot_pareto_front.best_candidate   = result_candidate
            plot_pareto_front.patience_counter = 0

    if plot_pareto_front.patience_counter >= args.patience:
        tqdm.write("Ran out of patience")
        meta_file.write("Ran out of patience\n")
        break
    elif complete_phi4.count >= args.budget:
        tqdm.write("Ran out of budget")
        break

    torch.cuda.empty_cache()
    gc.collect()

plot_pareto_front(population_data, fronts, iter_idx, args.meta_dir, final=True)


## Cell 12: Final Logging + Score Plots

In [ ]:
if 'wandb' in globals():
    wandb.log({"Final Best Candidate": result_candidate, "Final Objectives": result_objectives})
meta_file.write('\n')
tqdm.write("APICalls for search: " + str(complete_phi4.count))
meta_file.write("APICalls: " + str(complete_phi4.count) + '\n')
if 'wandb' in globals():
    wandb.log({"APICalls": complete_phi4.count})


def plot_best_candidate_scores(meta_dir, rouge_scores, bert_scores, bleu_scores,
                                summarization_scores, grammar_scores):
    iterations = list(range(len(rouge_scores)))
    metrics = [
        (rouge_scores,         'ROUGE Score',         'best_rouge_scores.png',         'Best ROUGE Scores'),
        (bert_scores,          'BERT Score',          'best_bert_scores.png',          'Best BERT Scores'),
        (bleu_scores,          'BLEU Score',          'best_bleu_scores.png',          'Best BLEU Scores'),
        (summarization_scores, 'Summarization Score', 'best_summarization_scores.png', 'Best Summarization Scores'),
        (grammar_scores,       'Grammar Score',       'best_grammar_scores.png',       'Best Grammar Scores'),
    ]
    for scores, ylabel, fname, wandb_key in metrics:
        plt.figure(figsize=(8, 6))
        plt.plot(iterations, scores, marker='o', linestyle='-')
        plt.xlabel('Iteration Number')
        plt.ylabel(ylabel)
        plt.title(f'Best Candidate {ylabel} vs Iteration (0 = Best Initial Candidate)')
        plt.grid(True)
        plt.xticks(iterations)
        plot_path = os.path.join(meta_dir, fname)
        plt.savefig(plot_path)
        plt.close()
        if 'wandb' in globals():
            wandb.log({wandb_key: wandb.Image(plot_path)})


plot_best_candidate_scores(
    args.meta_dir,
    best_rouge_scores, best_bert_scores, best_bleu_scores,
    best_summarization_scores, best_grammar_scores
)

if 'wandb' in globals():
    wandb.save(meta_path)
meta_file.close()
global_progress_bar.close()

tqdm.write(f"\nFinal Best Prompt:\n{result_candidate}")
tqdm.write(f"Final Objectives (Summarization, Grammar): {result_objectives}")
